# prep_01_zip_centroids
## Ohio Dental Clinic — Site Selection Analysis

**Purpose:**  
Cleans and standardizes the SimpleMaps U.S. ZIP code database for Ohio.  
Extracts latitude and longitude centroids for all Ohio ZIP codes used in **Haversine distance calculations** throughout the analysis.

---

| Item | Description |
|-----|-------------|
| **Input** | `simplemaps_uszips_raw.csv` |
| **Output** | `ohio_zip_centroids_cleaned.csv` |
| **Records** | ~1,232 Ohio ZIP codes |

In [3]:
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# PATHS 
# RAW_DATA_PATH can be updated depending on where you saved the raw files on your machine
RAW_DATA_PATH = r"C:\Users\mosun\Downloads\oh_clinic_rw_files"
OUTPUT_PATH = r"../data/cleaned"

# LOAD
print("Loading uszips.csv...")
zips_raw = pd.read_csv(os.path.join(RAW_DATA_PATH, "uszips.csv"), dtype={"zip": str})
zips_raw.columns = zips_raw.columns.str.strip().str.lower()
print(f"Raw file: {len(zips_raw):,} rows | Columns: {list(zips_raw.columns)}")

# Filter to Ohio only
ohio = zips_raw[zips_raw["state_id"] == "OH"].copy()
print(f"After Ohio filter: {len(ohio)} rows")

# Keep only true ZCTAs — removes PO boxes and non-standard ZIPs
# SimpleMaps stores zcta as the string "TRUE" not a boolean
ohio = ohio[ohio["zcta"].astype(str).str.upper() == "TRUE"].copy()
print(f"After ZCTA filter: {len(ohio)} rows")

# Rename lat/lng to latitude/longitude
ohio = ohio.rename(columns={"lat": "latitude", "lng": "longitude"})

# Ensure 5-digit ZIP with leading zeros
ohio["zip"] = ohio["zip"].astype(str).str.zfill(5)

# Keep only the 3 columns needed — one row per ZIP
ohio_zips = ohio[["zip", "latitude", "longitude"]].drop_duplicates(subset="zip")

# SAVE
ohio_zips.to_csv(os.path.join(OUTPUT_PATH, "ohio_zip_centroids_cleaned.csv"), index=False)
print(f"\nSaved: ohio_zip_centroids_cleaned.csv")
print(f"Rows: {len(ohio_zips)} | Columns: {list(ohio_zips.columns)}")
print(f"\nSample:")
print(ohio_zips.head(5).to_string(index=False))

Loading uszips.csv...
Raw file: 33,782 rows | Columns: ['zip', 'lat', 'lng', 'city', 'state_id', 'state_name', 'zcta', 'parent_zcta', 'population', 'density', 'county_fips', 'county_name', 'county_weights', 'county_names_all', 'county_fips_all', 'imprecise', 'military', 'timezone']
After Ohio filter: 1232 rows
After ZCTA filter: 1232 rows

Saved: ohio_zip_centroids_cleaned.csv
Rows: 1232 | Columns: ['zip', 'latitude', 'longitude']

Sample:
  zip  latitude  longitude
43001  40.08794  -82.61289
43002  40.05982  -83.17305
43003  40.41017  -82.96866
43004  40.01664  -82.80025
43005  40.28241  -82.26728
